# Design of a low pass filter that complies with EMC-regulations
Two approaches are given here.

## LTSpice approach
Advantage is that it can use the Wuerth models.
1. Model the current (noise) source as a square wave or as a wave according to DC/DC power handbook. With the following parameters:
  * Vin, Vout : for duty cycle
  * I_avg : average output (=input) current
  * ripple_factor
  * switching frequency
2. Determine input capacitor by checking RED-expert and finding the one with the lowest impedance at the switching frequency.
3. Do harmonic analysis or run FFT-analysis.
4. Check if you comply with the conducted emissions regulation with the given margin (ripple - 6dB < 45 dBµV).  If you do, you're done here.  If you don't, you'll have to add an LC-lowpass filter.
5. Determine the cut-off frequency of the LC-filter by determining how much attenuation you need at the switching frequency.  See the python code below.
6. Go to [RED-Expert EMI-filter designer](https://redexpert.we-online.com/we-redexpert/en/#/emifd-embedded) and select "CL" topology.  The tool will propose a filter.
7. Add the proposed L and C values to your simulation model and check if you comply.  If not, you might have to choose a 4th-order CL-CL model.

## Analytical approach
This doesn't use any online tools.  You do need the ESR and ESL values of the input capacitor.

In [5]:
import math
def calc_harmonics(Vin, Vout, I_out_avg, Z_C, f_sw):
    # Vin : Input voltage of the buck regulator in [V]
    # Vout : Output voltage of the buck regulator in [V]
    # I_out_avg : average output current of the regulator in [A]
    D = Vout / Vin
    V_C_dBµV = []
    for i in range(len(Z_C)):
        Iout_n = math.sqrt(2)*I_out_avg*abs(math.sin(math.pi * (i+1) * D)) / (math.pi * (i+1))
        V_Cn = Iout_n * Z_C[i]
        V_Cn_dBµV = 20 * math.log10(V_Cn/1e-6)
        V_C_dBµV.append(V_Cn_dBµV)
        # print(f"{V_Cn_dBµV:.0f} dBµV at {f_sw *(i+1)/1e3:.0f} kHz")
    return V_C_dBµV

def calc_capacitor_impedance(f, N_harmonics, ESR, ESL, C,):
    # f : base frequency in [Hz]
    # N_harmonics : number of harmonics (including base frequency) to consider.
    # ESR : ESR of the capacitor in [ohm]
    # ESL : ESL of the capacitor in [H]
    # C : capacitance value of the capacitor in [F]
    Z_C = []
    for i in range(1,N_harmonics+1):
        ω = 2 * math.pi * f * i
        Z = ESR + ω * ESL + 1 /(ω *C)
        Z_C.append(Z)
    return Z_C

def check_CE_limit(V_dB):
    # V_dB : voltage in dBµV
    CE_limit = 45 # dBµV
    for i in range(len(V_dB)):
        if V_dB[i] > CE_limit:
            print(f"CE limit exceeded at {f_sw *(i+1)/1e3:.0f} kHz: {V_dB[i]:.0f} dBµV")

Vin = 24
Vout = 5
Iout = 1
f_sw = 300e3
P = 2 # number of equal capacitors in parallel.

# These values can be derived from the impedance versus frequency charts of the manufacturers.
# Typical characteristic is V-shaped.
# C can be read on derived from side of V-shape (take DC-derating into account)
# ESR can be read at bottom of V-shape
# ESL can be derived from right side of V-shape
ESR = 1e-3 # ESR of a single capacitor
ESL = 1e-9 # ESL of a single capacitor
C = 10e-6 # capacitance of a single capacitor

Z_C = calc_capacitor_impedance(f_sw, 10, ESR / P, ESL / P , C * P)
check_CE_limit(calc_harmonics(Vin, Vout, Iout, Z_C, f_sw))

CE limit exceeded at 300 kHz: 78 dBµV
CE limit exceeded at 600 kHz: 71 dBµV
CE limit exceeded at 900 kHz: 65 dBµV
CE limit exceeded at 1200 kHz: 56 dBµV
CE limit exceeded at 1800 kHz: 55 dBµV
CE limit exceeded at 2100 kHz: 57 dBµV
CE limit exceeded at 2400 kHz: 55 dBµV
CE limit exceeded at 2700 kHz: 47 dBµV


In [2]:
# Calculate cut off frequency for a single stage LC-filter, given the required attenuation

def calc_cutoff(att_dB, f_sw_Hz):
    # att_dB : required attenuation in [dB]
    # f_sw_Hz : switching frequency in [Hz]
    # return : cut-off frequency in [Hz]
    f_co = f_sw_Hz / 10**(att_dB / 40)
    return f_co

f_sw_Hz = 300e3 # switching frequency [Hz]
req_att = 30 # required attenuation [dB]
print(f"Cut-off frequency : {calc_cutoff(req_att, f_sw_Hz)/1e3:.0f} kHz")

Cut-off frequency : 53 kHz
